# Setup

In [5]:
import numpy as np
import torch
import torch.nn as nn
import random

from ssl_model.models import SpectralBoundaryEncoder
from torch.utils.data import DataLoader

In [6]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
hop_len = 48
sample_rate = 48000

In [12]:
def load_embedding_model(path: str, config_path: str) -> torch.nn.Module:
    with open(config_path, "r") as f:
        config = json.load(f)
    model = SpectralBoundaryEncoder(**config["model"])
    model.load_state_dict(torch.load(path, map_location=device, weights_only=False))
    model.eval().to(device)
    return model


### ---------- SPECIFY EMBEDDING MODEL -----------
EMBEDDING_MODEL_PATH = (
    "watkins/models/final_model.pt"
    # "dominica/models/final_model.pt"
    # "both/models/final_model.pt"
)
CONFIG_PATH = "configs/pipeline.json"

embedding_model = load_embedding_model(EMBEDDING_MODEL_PATH, CONFIG_PATH)

In [26]:
import os
import random
import json
import pandas as pd
import torch
from torch.utils.data import Dataset
import librosa
import soundfile as sf
from pathlib import Path

import warnings

# Suppress the specific FutureWarning from librosa
warnings.filterwarnings("ignore", category=FutureWarning, module="librosa")
warnings.filterwarnings(
    "ignore", message="PySoundFile failed. Trying audioread instead."
)


class SpermWhaleClicks(Dataset):
    def __init__(
        self,
        base_path: str = "../../data/wavs48khz_watkins/*.wav",
        selections_path: str = "../../data/selections/*.txt",
        window: float = 0.5,
        window_pad: int = 136,
        sample_rate: int = 48000,
        seed: int = 42,
        epsilon: float = 2e-6,
        hop_len: int = 48,
        augment: bool = False,
    ) -> None:
        """
        Initializes the dataset for Sperm Whale Clicks.

        Args:
            base_path (str): Glob pattern to locate WAV files.
            window (float): Active window duration in seconds.
            window_pad (int): Padding (in frames) added to the window.
            sample_rate (int): Sampling rate for audio.
            seed (int): Random seed for reproducibility.
            epsilon (float): Small value to prevent boundary issues.
            hop_len (int): Hop length (in frames) used for splitting the active window into embeddings.
        """
        self.window = window
        self.window_pad = window_pad
        self.sample_rate = sample_rate
        self.seed = seed
        self.epsilon = epsilon
        self.hop_len = hop_len
        self.augment = augment

        self.device = device or torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        # Initialize randomness
        self._set_seeds()

        # Data loading
        self.annotations = self._load_annotations(selections_path, base_path)
        self.files = self._get_files(base_path)

        # Sample precomputation
        self.positive_samples = self._precompute_samples()
        self.total_samples = len(self.positive_samples)

    def _set_seeds(self) -> None:
        """Initialize all relevant random seeds."""
        np.random.seed(self.seed)
        random.seed(self.seed)
        torch.manual_seed(self.seed)

    def _get_files(self, pattern: str) -> list[Path]:
        """Resolve file pattern to sorted list of Path objects."""
        path = Path(pattern)
        return sorted(path.parent.glob(path.name))

    def _load_annotations(self, path: str, base_path: str) -> dict[str, list[float]]:
        # Load annotations
        selections = self._get_files(path)
        annotations = {}
        for path in selections:
            name = Path(path).name.split(".")[0]
            df = pd.read_csv(path, delimiter="\t")
            click_times = list(set(df["Begin Time (s)"].tolist()))
            end_times = list(set(df["End Time (s)"].tolist()))
            annotations[base_path[:-5] + name + ".wav"] = list(
                zip(click_times, end_times)
            )
        return annotations

    def _precompute_samples(
        self,
    ) -> tuple[list[tuple[Path, float]], list[tuple[Path, float]]]:
        positive_samples = []
        for wav_path in self.files:
            file_name = wav_path.as_posix()
            if file_name in self.annotations:
                for click_time, _ in self.annotations[file_name]:
                    # Adjust start time so that click is inside the window
                    start_time = max(0.0, click_time - self.epsilon)
                    # Ensure window fits within file duration
                    dur = sf.info(file_name).duration
                    if dur < self.window:
                        continue
                    max_start = dur - (self.window + self.epsilon)
                    if start_time > max_start:
                        start_time = max_start
                    positive_samples.append((file_name, start_time))

        return positive_samples

    def __len__(self) -> int:
        return self.total_samples

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        wav, start_time = self.positive_samples[idx]

        file_info = sf.info(wav)
        dur = file_info.duration

        # Introduce a small random offset to the start_time
        # Add random offset if augmentation is enabled
        if self.augment:
            max_offset = self.window - self.epsilon  # Maximum offset in seconds
            offset = random.uniform(-max_offset, max_offset)
            start_time = start_time + offset
            # Ensure we stay within valid bounds
            start_time = max(0.0, min(start_time, dur - self.window - self.epsilon))

        # Load the audio snippet with extra padding.
        total_duration = self.window + self.window_pad / self.sample_rate + self.epsilon

        x, _ = librosa.load(
            wav,
            offset=start_time,
            duration=self.window + self.epsilon,
            sr=self.sample_rate,
        )

        # Compute total number of frames (samples).
        window_frames = int(self.window * self.sample_rate) + self.window_pad
        x = librosa.util.fix_length(data=x, size=window_frames)
        # Convert to tensor with a channel dimension.
        x = torch.tensor(x, dtype=torch.float32).unsqueeze(dim=0)
        # At this point, x has shape [1, window_frames].

        # --- Build per-embedding labels ---
        # We consider only the "active" window (without the extra padding)
        active_frames = int(self.window * self.sample_rate)
        # Compute number of embeddings (this may vary if the active window length changes)
        n_embeddings = active_frames // self.hop_len + 1
        # Prepare a label vector (initially all zeros) of shape [n_embeddings]
        labels = np.zeros(n_embeddings, dtype=np.float32)

        # Calculate the center time (in seconds) for each embedding relative to the start of the snippet.
        # Here, each embedding is assumed to cover self.hop_len frames,
        # and its center is at: start_time + (i * hop_len + hop_len/2) / sample_rate.
        embedding_times = np.array(
            [
                start_time + (i * self.hop_len + self.hop_len / 2) / self.sample_rate
                for i in range(n_embeddings)
            ]
        )

        # --- Determine which embeddings are positive ---
        # Extract the base name (e.g., "19620917a") from the wav filename.
        base_name = os.path.splitext(os.path.basename(wav))[0]
        selection_file = f"../data/selections/{base_name}.selections.txt"
        if not os.path.exists(selection_file):
            selection_file = f"../data/selections/{base_name}.q.selections.txt"

        if os.path.exists(selection_file):
            # Read the selection file.
            df = pd.read_csv(selection_file, sep="\t")
            # Assuming every other row corresponds to a unique selection.
            df = df.iloc[::2].reset_index(drop=True)
            begin_times = df["Begin Time (s)"].to_numpy()
            end_times = df["End Time (s)"].to_numpy()

            # For each event, mark the embeddings whose center falls within the event interval.
            for b, e in zip(begin_times, end_times):
                # Find indices where the embedding center falls in [b, e].
                indices = np.where(
                    (embedding_times >= b - 0.0005) & (embedding_times <= b + 0.0005)
                )[0]
                labels[indices] = 1.0
        # If no selection file exists, labels remain all zeros.

        # Convert labels to a tensor of shape [n_embeddings].
        labels_tensor = torch.tensor(labels, dtype=torch.float32)

        return x, labels_tensor

In [27]:
import scipy.io as sio

class DominicaClicks(SpermWhaleClicks):
    def __init__(
        self,
        base_path: str = "../data/Dominica_dataset/Signal_parts/*.wav",
        annotations_path: str = "../data/Dominica_dataset/Annotations_Dominica.mat",
        window: float = 0.5,
        window_pad: int = 136,
        sample_rate: int = 48000,
        epsilon: float = 2e-6,
        hop_len: int = 48,
        seed: int = 42,
        augment: bool = False,
    ) -> None:
        # Configuration parameters
        self.window = window
        self.window_pad = window_pad
        self.sample_rate = sample_rate
        self.epsilon = epsilon
        self.hop_len = hop_len
        self.seed = seed
        self.augment = augment

        self.device = device or torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        # Initialize randomness
        self._set_seeds()

        # Data loading
        self.annotations = self._load_annotations(Path(annotations_path), base_path)
        self.files = self._get_files(base_path)

        # Sample precomputation
        self.positive_samples = self._precompute_samples()
        self.total_samples = len(self.positive_samples)

    def _load_annotations(self, path: Path, base_path: str) -> dict[str, list[float]]:
        """Load and parse MATLAB annotations file."""
        if not path.exists():
            raise FileNotFoundError(f"Annotations file {path} not found")

        mat_data = sio.loadmat(str(path))["Annotations_Dominica"]
        return {
            f"{base_path[:-5]}{entry[0][0].item()}": entry[1][0].tolist() if entry[1].size > 0 else []
            for entry in mat_data[1:]  # Skip header
        }

    def _precompute_samples(
        self,
    ) -> tuple[list[tuple[Path, float]], list[tuple[Path, float]]]:
        positive_samples = []
        for wav_path in self.files:
            file_name = wav_path.as_posix()
            if file_name in self.annotations:
                for click_time in self.annotations[file_name]:
                    # Adjust start time so that click is inside the window
                    start_time = max(0.0, click_time - self.epsilon)
                    # Ensure window fits within file duration
                    dur = sf.info(file_name).duration
                    if dur < self.window:
                        continue
                    max_start = dur - (self.window + self.epsilon)
                    if start_time > max_start:
                        start_time = max_start
                    positive_samples.append((file_name, start_time))

        return positive_samples

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        wav, start_time = self.positive_samples[idx]

        file_info = sf.info(wav)
        dur = file_info.duration

        # Introduce a small random offset to the start_time
        # Add random offset if augmentation is enabled
        if self.augment:
            max_offset = self.window - self.epsilon  # Maximum offset in seconds
            offset = random.uniform(-max_offset, max_offset)
            start_time = start_time + offset
            # Ensure we stay within valid bounds
            start_time = max(0.0, min(start_time, dur - self.window - self.epsilon))

        # Load the audio snippet with extra padding.
        total_duration = self.window + self.window_pad / self.sample_rate + self.epsilon

        x, _ = librosa.load(
            wav,
            offset=start_time,
            duration=self.window + self.epsilon,
            sr=self.sample_rate,
        )

        # Compute total number of frames (samples).
        window_frames = int(self.window * self.sample_rate) + self.window_pad
        x = librosa.util.fix_length(data=x, size=window_frames)
        # Convert to tensor with a channel dimension.
        x = torch.tensor(x, dtype=torch.float32).unsqueeze(dim=0)
        # At this point, x has shape [1, window_frames].

        # --- Build per-embedding labels ---
        # We consider only the "active" window (without the extra padding)
        active_frames = int(self.window * self.sample_rate)
        # Compute number of embeddings (this may vary if the active window length changes)
        n_embeddings = active_frames // self.hop_len + 1
        # Prepare a label vector (initially all zeros) of shape [n_embeddings]
        labels = np.zeros(n_embeddings, dtype=np.float32)

        # Calculate the center time (in seconds) for each embedding relative to the start of the snippet.
        # Here, each embedding is assumed to cover self.hop_len frames,
        # and its center is at: start_time + (i * hop_len + hop_len/2) / sample_rate.
        embedding_times = np.array(
            [
                start_time + (i * self.hop_len + self.hop_len / 2) / self.sample_rate
                for i in range(n_embeddings)
            ]
        )

        # --- Determine which embeddings are positive ---
        # Extract the base name (e.g., "19620917a") from the wav filename.
        begin_times = self.annotations[wav]
        begin_times = np.array(begin_times)
        for b in begin_times:
            # Find indices where the embedding center falls in [b, e].
            indices = np.where(
                (embedding_times >= b - 0.0005) & (embedding_times <= b + 0.0005)
            )[0]
            labels[indices] = 1.0
        # If no selection file exists, labels remain all zeros.

        # Convert labels to a tensor of shape [n_embeddings].
        labels_tensor = torch.tensor(labels, dtype=torch.float32)

        return x, labels_tensor

In [28]:
with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

dataset_params = config["dataset"]

In [29]:
# Load dataset parameters from config
with open("configs/pipeline.json", "r") as f:
    config = json.load(f)
    dataset_params = config["dataset"]
    training_params = config["training"]

In [30]:
from torch.utils.data import ConcatDataset, random_split

BATCH_SIZE = 32
train_size = 0.85
val_size = 1 - train_size

selection_dataset = SpermWhaleClicks(
    sample_rate=sample_rate,
    hop_len=hop_len,
    augment=True,
)
train_selection_dataset, val_selection_dataset = random_split(selection_dataset, [train_size, val_size], torch.Generator().manual_seed(42))

dominica_dataset = DominicaClicks(
    base_path="../../data/wavs48khz_dominica/*.wav",
    annotations_path="../../data/Dominica_dataset/Annotations_Dominica.mat",
    sample_rate=sample_rate,
    hop_len=hop_len,
    augment=True,
)
train_dominica_dataset, val_dominica_dataset = random_split(dominica_dataset, [train_size, val_size], torch.Generator().manual_seed(42))

train_dataset = ConcatDataset([
    train_selection_dataset,
    train_dominica_dataset,
])
val_dataset = ConcatDataset([
    val_selection_dataset,
    val_dominica_dataset,
])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [31]:
len(train_selection_dataset), len(train_dominica_dataset), len(train_dataset)

(1596, 24055, 25651)

In [32]:
len(val_selection_dataset), len(val_dominica_dataset), len(val_dataset)

(281, 4245, 4526)

In [33]:
i = 0
for x, y in train_loader:
    # print(i)
    # i+=1
    print(x.shape)
    print(y.shape)
    break
# print(64 * 501)

torch.Size([32, 1, 24136])
torch.Size([32, 501])


In [34]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = torch.nn.functional.binary_cross_entropy_with_logits(
            inputs, targets, reduction="none"
        )
        # p if target = 1, 1-p if target = 0
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss

In [35]:
import torch
from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryF1Score
from torchmetrics import MetricCollection
from sklearn.metrics import precision_recall_curve, auc, f1_score


class MetricManager:
    def __init__(
        self,
        device: str = "cuda" if torch.cuda.is_available() else "cpu",
        threshold: float = 0.45,
        selected_metrics: list = None,
        dynamic_threshold: bool = False,
    ):
        """
        Args:
            device (str): Device to run the metrics on.
            threshold (float): Initial threshold for binary metrics.
            selected_metrics (list): List of metric names to compute.
                Options can include: "precision", "recall", "f1", "pr_auc", "r_value".
                If None, defaults to ["precision", "recall", "f1"].
            dynamic_threshold (bool): If True, dynamically search for the threshold (nearest 5 candidates with step 0.01)
                                      that maximizes F1 score, using the current epoch's predictions.
        """
        self.device = device
        self.threshold = threshold
        self.dynamic_threshold = dynamic_threshold

        if selected_metrics is None:
            selected_metrics = ["precision", "recall", "f1"]
        self.selected_metrics = selected_metrics

        metric_dict = {}
        if "precision" in selected_metrics:
            metric_dict["precision"] = BinaryPrecision(threshold=threshold)
        if "recall" in selected_metrics:
            metric_dict["recall"] = BinaryRecall(threshold=threshold)
        if "f1" in selected_metrics:
            metric_dict["f1"] = BinaryF1Score(threshold=threshold)
        # Additional torchmetrics can be added here if needed.

        self.metrics = MetricCollection(metric_dict).to(self.device)

        # These two are computed externally (if selected).
        self.compute_pr_auc = "pr_auc" in self.selected_metrics
        self.compute_r_value = "r_value" in self.selected_metrics

        # For PR-AUC and threshold optimization, accumulate predictions and targets.
        self.all_preds = []
        self.all_targets = []

        # Dummy buffer to simulate nn.Module behavior for device placement.
        self.register_buffer("dummy", torch.tensor(0))

    def update(self, preds: torch.Tensor, targets: torch.Tensor):
        """Update internal metrics and accumulate predictions for PR-AUC and threshold optimization."""
        self.metrics.update(preds, targets)
        self.all_preds.append(preds.detach().cpu())
        self.all_targets.append(targets.detach().cpu())

    def optimize_threshold(self):
        """
        Searches for the best threshold within a small neighborhood of the current threshold.
        It checks the nearest 5 thresholds with a step of 0.01 (i.e. a window of ±0.02 around the current threshold),
        and updates the threshold based on the F1 score.
        """
        current_thresh = self.threshold
        lower = max(0.0, current_thresh - 0.02)
        upper = min(1.0, current_thresh + 0.02)
        candidate_thresholds = np.linspace(lower, upper, 5)

        combined_preds = torch.cat(self.all_preds).numpy()
        combined_targets = torch.cat(self.all_targets).numpy()

        best_thresh = current_thresh
        best_f1 = 0.0
        for thresh in candidate_thresholds:
            preds_bin = (combined_preds >= thresh).astype(np.float32)
            current_f1 = f1_score(combined_targets, preds_bin, zero_division=0)
            if current_f1 > best_f1:
                best_f1 = current_f1
                best_thresh = thresh

        # Update threshold in this manager.
        self.threshold = best_thresh
        # Update threshold for all torchmetrics that support it.
        for metric in self.metrics.values():
            if hasattr(metric, "threshold"):
                metric.threshold = best_thresh

        return best_thresh, best_f1

    def compute(self) -> dict:
        """Compute and return selected metrics.
        Optionally, dynamically update the threshold before computing.
        Returns only the metrics that are selected.
        """

        # Optionally update the threshold dynamically
        if self.dynamic_threshold and len(self.all_preds) > 0:
            self.optimize_threshold()

        metrics = self.metrics.compute()
        output = {k: v.item() if hasattr(v, "item") else v for k, v in metrics.items()}

        # Compute PR-AUC if requested.
        if self.compute_pr_auc:
            try:
                concatenated_targets = torch.cat(self.all_targets).numpy()
                concatenated_preds = torch.cat(self.all_preds).numpy()
                precisions, recalls, _ = precision_recall_curve(
                    concatenated_targets, concatenated_preds
                )
                output["pr_auc"] = auc(recalls, precisions)
            except ValueError:
                output["pr_auc"] = float("nan")

        # Compute r_value if requested.
        if self.compute_r_value:
            combined_preds = torch.cat(self.all_preds)
            combined_targets = torch.cat(self.all_targets)
            num_positives = int(combined_targets.sum().item())
            if num_positives > 0:
                _, top_indices = torch.topk(combined_preds, k=num_positives)
                output["r_value"] = combined_targets[top_indices].float().mean().item()
            else:
                output["r_value"] = 0.0

        # Clear accumulated predictions and targets for the next epoch.
        # self.all_preds.clear()
        # self.all_targets.clear()

        return output

    def reset(self):
        """Reset all internal metrics and accumulated predictions/targets."""
        self.metrics.reset()
        self.all_preds.clear()
        self.all_targets.clear()

    def register_buffer(self, name: str, tensor: torch.Tensor):
        """Simulate nn.Module.register_buffer for proper device placement."""
        setattr(self, name, tensor)

In [36]:
import torch
import torch.nn as nn
import pytorch_lightning as pl


class LitEmbeddingLSTMModel(pl.LightningModule):
    def __init__(self, embedding_model, learning_rate=1e-3):
        super().__init__()
        self.learning_rate = learning_rate

        # Example: assume self.embedding_model and some new layers are defined
        self.embedding_model = embedding_model  # Pre-trained
        # Freeze all parameters of the embedding model
        # for name, param in self.embedding_model.named_parameters():
        #     param.requires_grad = False
        # Unfreeze the last N parameters
        # unfreeze_count = 10
        # params = list(self.embedding_model.parameters())
        # for param in params[-unfreeze_count:]:
        #     param.requires_grad = True

        # New layers (replace with your actual model)
        self.lstm = nn.LSTM(
            input_size=32,
            hidden_size=64,
            num_layers=4,
            
            batch_first=True,
            bidirectional=True,
            dropout=0.3 if 2 > 1 else 0.0,
        )
        self.fc = nn.Linear(2 * 64, 1)

        # Loss function (FocalLoss, as in the CNN version)
        self.loss_fn = FocalLoss(alpha=0.3, gamma=3.5, reduction="mean")

        # Metrics
        self.threshold = 0.5
        self.val_metrics = MetricManager(
            threshold=self.threshold, dynamic_threshold=False
        )
        # self.train_metrics = MetricManager(
        #     threshold=self.threshold, dynamic_threshold=False
        # )
        self.best_val_f1 = 0.0

    def forward(self, x):
        # Get embeddings from the embedding model (shape [B, T, 32])
        embedding = self.embedding_model(x)
        # Pass embeddings through LSTM and FC layers (example)
        lstm_out, _ = self.lstm(embedding)
        logits = self.fc(lstm_out)
        logits = logits.squeeze(-1)  # [B, T]
        return logits

    def training_step(self, batch, batch_idx):
        inputs, targets = batch  # (inputs, targets)
        logits = self(inputs)
        loss = self.loss_fn(logits, targets)

        # predictions = torch.sigmoid(logits)
        # self.train_metrics.update(predictions.flatten(), targets.flatten())

        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, targets = batch
        logits = self(inputs)
        loss = self.loss_fn(logits, targets)
        predictions = torch.sigmoid(logits)

        # Update metrics
        self.val_metrics.update(predictions.flatten(), targets.flatten())

        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    # def on_validation_epoch_end(self):
    #     # Compute all metrics
    #     metrics = self.val_metrics.compute()
    #     current_f1 = metrics["f1"]

    #     # Update best F1 if current is better
    #     # if current_f1 > self.best_val_f1 and len(self.val_metrics.all_preds) > 0:
    #     #     self.threshold, current_f1 = self.val_metrics.optimize_threshold()
    #     #     self.best_val_f1 = current_f1
    #     #     self.print(f"New best F1: {self.best_val_f1:.4f}")

    #     # Log metrics
    #     # for name, value in metrics.items():
    #     self.log(f"val_f1", current_f1, prog_bar="f1")
    #     # self.log(f"best_thresh", self.threshold, prog_bar="threshold")

    #     # Reset metrics
    #     self.val_metrics.reset()

    # def on_train_epoch_end(self):
    #     # Compute all metrics
    #     metrics = self.train_metrics.compute()
    #     for name, value in metrics.items():
    #         self.log(f"train_{name}", value, prog_bar=f"train_{name}")    
    #     # Reset metrics
    #     self.train_metrics.reset()

    def on_validation_epoch_end(self):
        # Compute all metrics
        metrics = self.val_metrics.compute()
        for name, value in metrics.items():
            self.log(f"val_{name}", value, prog_bar=f"val_{name}")
        # Reset metrics
        self.val_metrics.reset()

    def configure_optimizers(self):
        # Separate parameters into two groups: embedding and new layers.
        embedding_params = []
        new_params = []
        for name, param in self.named_parameters():
            if "embedding_model" in name:
                embedding_params.append(param)
            else:
                new_params.append(param)

        optimizer = torch.optim.AdamW(
            [
                {"params": embedding_params, "lr": self.learning_rate * 0.1},
                {"params": new_params, "lr": self.learning_rate},
            ],
            weight_decay=1e-4,
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=5, eta_min=1e-7
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
                "frequency": 1,
            },
        }

    def configure_gradient_clipping(
        self, optimizer, gradient_clip_val, gradient_clip_algorithm
    ):
        # Clip gradients to improve stability, if desired.
        torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=1.0)

In [37]:
# Instantiate the Lightning model
model = LitEmbeddingLSTMModel(embedding_model)

In [38]:
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint

# Initialize logger
tb_logger = TensorBoardLogger(
    save_dir="lstm-detector/", name="lstm_model", log_graph=True  # Log model computational graph
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_f1",
    mode="max",
    save_top_k=1,
    filename="best-lstm-detector",
    save_weights_only=False,
    verbose=False,
)

# Initialize Trainer
trainer = pl.Trainer(
    logger=tb_logger,
    max_epochs=training_params.get("max_epochs", 50),
    accelerator=device,
    devices=1,
    check_val_every_n_epoch=5,
    callbacks=[checkpoint_callback],
    enable_checkpointing=True,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [ ]:
# Start training
trainer.fit(model, train_loader, val_loader)

In [ ]:
import numpy as np

best_model = LitEmbeddingLSTMModel.load_from_checkpoint(
    checkpoint_callback.best_model_path, embedding_model=embedding_model
)

best_model.eval()

best_model.to("cpu")


# Define a function to compute F1 score for different thresholds
def compute_f1_scores(model, val_loader, thresholds):
    f1_scores = []
    for threshold in thresholds:
        f1_metric = BinaryF1Score(threshold=threshold)
        for inputs, targets in val_loader:
            logits = model(inputs)
            predictions = torch.sigmoid(logits)
            f1_metric.update(predictions.flatten(), targets.flatten())
        f1_scores.append(f1_metric.compute().item())
        print(f"F1 Score: {f1_scores[-1]} for threshold {threshold}")
    return f1_scores


# Define the range of thresholds to test
thresholds = np.arange(0.25, 0.55, 0.01)

# Compute F1 scores for each threshold
f1_scores = compute_f1_scores(best_model, val_loader, thresholds)

# Find the best threshold
best_threshold = thresholds[np.argmax(f1_scores)]
best_f1_score = max(f1_scores)

print(f"Best Threshold: {best_threshold}")
print(f"Best F1 Score: {best_f1_score}")

In [ ]:
metrics = MetricManager(selected_metrics = ["precision", "recall", "f1", "pr_auc", "r_value"])

for inputs, targets in val_loader:
    logits = model(inputs)
    predictions = torch.sigmoid(logits)
    metrics.update(predictions.flatten(), targets.flatten())
results = metrics.compute()
metrics.reset()

In [ ]:
results

{'f1': 0.9149772524833679,
 'precision': 0.9439902901649475,
 'recall': 0.8876944184303284,
 'pr_auc': 0.9657859344271219,
 'r_value': 0.9115968942642212}